# JobGuard AI - Milestone 7: DistilBERT (Google Colab version)

**Goal:** fine-tune a pretrained **DistilBERT** transformer to classify job postings as legitimate (0) or fraudulent (1), using the same validation-based protocol as Milestones 5 and 6.

This is the **Colab (CUDA GPU) version** of `notebooks/06_distilbert.ipynb`. Only the *environment* changed (GPU setup, file upload, self-contained helper code). The **methodology is identical**: same model, same data, same internal split, same `max_length`, epochs, learning rate, batch sizes, metrics, and the same baseline + class-weighted experiment design.

| Data | Used for |
|---|---|
| **Model-training subset** (80% of `train.csv`) | teaching (fine-tuning) the model |
| **Validation subset** (20% of `train.csv`) | **choosing between settings and comparing models** |
| **Test set** (`test.csv`) | **NOT uploaded and NOT used in this notebook.** Reserved for Milestone 8 |

**Plan**
1. Set up Colab (install packages, confirm the GPU), upload `train.csv`, recreate the same internal split and prove it is identical
2. Understand DistilBERT and tokenization (it does **not** use TF-IDF)
3. Fine-tune a baseline DistilBERT and one class-weighted version
4. Pick the better one **on validation data**
5. Save the models, retrain the selected setting on the full `train.csv`, and package everything for download

**No test-set metrics are calculated anywhere in this notebook, and `test.csv` is never needed.**

## 0. Colab quick start

1. **Turn on the GPU:** menu *Runtime -> Change runtime type -> Hardware accelerator: GPU* (a **T4** is fine) -> *Save*.
2. **Run the cells from top to bottom** (*Runtime -> Run all*, or one at a time with Shift+Enter).
3. When the upload cell (Section 2) asks for files, choose **`train.csv`**. You can *optionally* also choose `model_validation_comparison.csv` (a small file of the earlier models' validation results) to get the four-model comparison here; without it, that one comparison is skipped and can be made locally.
4. **Never upload `test.csv`.** This notebook refuses to continue if it sees one.
5. The last cell downloads two zip files. Colab's storage is temporary, so download them before closing the session.

The next cell installs the library versions this experiment was developed with. If Colab then asks you to *restart the session*, do so and run the notebook again from the top.

In [ ]:
# Install the exact versions used during development (torch, scikit-learn, pandas and matplotlib already come with Colab).
!pip install -q "transformers==5.20.2" "accelerate==1.15.0"

In [ ]:
# Detect the GPU. Fine-tuning needs a CUDA GPU, so we stop with a clear message if there is none.
import torch

print("PyTorch version:", torch.__version__)
print("CUDA GPU available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. In Colab choose: Runtime -> Change runtime type -> Hardware accelerator: GPU "
                       "(T4), press Save, and then run the notebook again from the top.")

device_name = "cuda"
gpu_name = torch.cuda.get_device_name(0)
print("GPU name:", gpu_name)
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.1f} GB")

## 1. Imports, output folders and helper code

New libraries for this milestone: **torch** (deep-learning engine), **transformers** (Hugging Face models and the `Trainer` helper) and **accelerate** (used by `Trainer` behind the scenes). The pretrained checkpoint is `distilbert-base-uncased`.

Everything this notebook produces goes into one folder, `jobguard_outputs/`, laid out like the local project (`results/figures/`, `results/metrics/`, `models/`), so it can be copied straight back into your JobGuard AI project.

To keep the notebook **self-contained**, the two small helper pieces it needs from the local project are copied in below, unchanged: the metric/confusion-matrix code from `src/evaluation.py`, and the `create_combined_text` function from `src/preprocessing.py` (used only for the three demonstration examples).

In [ ]:
import hashlib
import math
import shutil
import tempfile
import time
from pathlib import Path

import accelerate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import transformers
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

# All outputs are written here (a relative folder, so no machine-specific paths are needed)
OUTPUT_ROOT = Path("jobguard_outputs")
FIGURES_DIR = OUTPUT_ROOT / "results" / "figures"
METRICS_DIR = OUTPUT_ROOT / "results" / "metrics"
MODELS_DIR = OUTPUT_ROOT / "models"
for folder in [FIGURES_DIR, METRICS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 256
SEED = 42

print("torch", torch.__version__, "| transformers", transformers.__version__, "| accelerate", accelerate.__version__)

In [ ]:
# ---- Copied unchanged from src/evaluation.py: the fraudulent class (label 1) is the "positive" class ----
def calculate_metrics(y_true, y_pred):
    true_neg, false_pos, false_neg, true_pos = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Fraud Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Fraud Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Fraud F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "True Negatives": int(true_neg),
        "False Positives": int(false_pos),
        "False Negatives": int(false_neg),
        "True Positives": int(true_pos),
    }


def plot_confusion_matrix(y_true, y_pred, title, save_path):
    matrix = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)

    for row in range(2):
        for col in range(2):
            # white text on dark cells, black text on light cells
            text_color = "white" if matrix[row, col] > matrix.max() / 2 else "black"
            ax.text(col, row, f"{matrix[row, col]:,}", ha="center", va="center",
                    fontsize=14, color=text_color)

    ax.set_xticks([0, 1], labels=["Predicted Legitimate", "Predicted Fraudulent"])
    ax.set_yticks([0, 1], labels=["Actual Legitimate", "Actual Fraudulent"])
    ax.set_title(title)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# ---- Copied unchanged from src/preprocessing.py (only used for the 3 demonstration examples) ----
def create_combined_text(df):
    return (
        "TITLE: " + df["title"] + "\n"
        + "COMPANY PROFILE: " + df["company_profile"] + "\n"
        + "DESCRIPTION: " + df["description"] + "\n"
        + "REQUIREMENTS: " + df["requirements"] + "\n"
        + "BENEFITS: " + df["benefits"]
    )

## 2. Upload `train.csv` and recreate the internal split

Run the next cell and, when the **Choose files** button appears, select **`train.csv`** from your local `data/processed/` folder (and optionally `model_validation_comparison.csv` from `results/metrics/`). **Do not select `test.csv`.**

Then we do two safety checks:
- **`test.csv` must not be present.** If it was uploaded by mistake, it is deleted and the notebook stops.
- **This must be the same `train.csv`** as in the local project. Its MD5 fingerprint, and the fingerprints of the model-training and validation row indices produced by the split, are compared with values that were **computed by code from the local project** when this notebook was generated (not typed by hand).

In [ ]:
TRAIN_FILE = Path("train.csv")
EARLIER_RESULTS_FILE = Path("model_validation_comparison.csv")   # optional: earlier models' VALIDATION results

if not TRAIN_FILE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("train.csv was not found. Put train.csv in the same folder as this notebook "
                                "(or run this notebook in Google Colab, where an upload button appears).")
    print("Choose train.csv (and, optionally, model_validation_comparison.csv). Do NOT choose test.csv.")
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().startswith("test"):
            Path(name).unlink()
            raise ValueError("test.csv must NOT be used in Milestone 7. The uploaded file was deleted from this Colab "
                             "session. Run this cell again and upload only train.csv.")

if any(path.name.lower().startswith("test") for path in Path(".").glob("*.csv")):
    raise ValueError("A test file is present in the working folder. Milestone 7 must not use it. Delete it and run again.")
if not TRAIN_FILE.exists():
    raise FileNotFoundError("train.csv is still missing. Run this cell again and choose train.csv.")

HAVE_EARLIER_RESULTS = EARLIER_RESULTS_FILE.exists()
print("train.csv found:", TRAIN_FILE.exists())
print("model_validation_comparison.csv found (optional):", HAVE_EARLIER_RESULTS)

In [ ]:
# Values computed by code from the local project when this notebook was generated
EXPECTED = {
    "train_csv_md5": "0a5b5a32b8d08433aa9858ac2ec5eb71",
    "train_csv_rows": 12852,
    "model_train_rows": 10281,
    "model_train_fraud": 459,
    "validation_rows": 2571,
    "validation_fraud": 115,
    "model_train_index_md5": "3fc59e593da1624ac69c9d0c4f155348",
    "validation_index_md5": "ca1d771819f17921aeb8350c8f3f01b0"
}


def file_fingerprint(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def index_fingerprint(index):
    return hashlib.md5(",".join(str(i) for i in sorted(index)).encode()).hexdigest()


train_csv_fingerprint_at_start = file_fingerprint(TRAIN_FILE)
assert train_csv_fingerprint_at_start == EXPECTED["train_csv_md5"], \
    "This train.csv is not identical to the local project's train.csv. Upload data/processed/train.csv."
print("PASS: the uploaded train.csv is byte-for-byte identical to the local project's train.csv")

In [ ]:
train = pd.read_csv(TRAIN_FILE)
X = train["combined_text"]
y = train["fraudulent"]

(X_model_train, X_validation,
 y_model_train, y_validation) = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("train.csv shape:", train.shape)
print(f"Model-training rows: {len(X_model_train):,}   fraudulent: {int(y_model_train.sum())} "
      f"({y_model_train.mean() * 100:.2f}%)")
print(f"Validation rows:     {len(X_validation):,}   fraudulent: {int(y_validation.sum())} "
      f"({y_validation.mean() * 100:.2f}%)")

shared_texts = set(X_model_train) & set(X_validation)
print("Identical combined_text values in both subsets:", len(shared_texts))
assert len(shared_texts) == 0

## 3. Confirm this is exactly the same split as the local project

The row counts and the fingerprints of *which rows* went into each subset must match the values computed from the local project. If they match, the model-training and validation postings here are the same ones used by Logistic Regression, Naive Bayes and XGBoost.

In [ ]:
actual = {
    "train_csv_md5": train_csv_fingerprint_at_start,
    "train_csv_rows": len(train),
    "model_train_rows": len(X_model_train),
    "model_train_fraud": int(y_model_train.sum()),
    "validation_rows": len(X_validation),
    "validation_fraud": int(y_validation.sum()),
    "model_train_index_md5": index_fingerprint(X_model_train.index),
    "validation_index_md5": index_fingerprint(X_validation.index),
}
for key in EXPECTED:
    status = "OK " if actual[key] == EXPECTED[key] else "MISMATCH"
    print(f"{status} {key:24s} expected {EXPECTED[key]} | actual {actual[key]}")

split_matches_previous = (actual == EXPECTED)
x_is_all_of_train_csv = (len(X) == len(train))
assert split_matches_previous
print()
print("PASS: identical train.csv, identical model-training rows and identical validation rows")

## 4. What is DistilBERT, and how is it different from TF-IDF?

**BERT** is a **transformer** language model. It was first **pretrained** on a huge amount of text (books and Wikipedia) so that it learned how language works: grammar, word meanings, and how words relate to each other.

**DistilBERT** is a smaller, faster version of BERT. It is trained to imitate BERT ("distillation"), keeps most of BERT's language understanding, and is roughly 40% smaller and noticeably faster. That makes it practical for a class project.

**How is this different from TF-IDF models?**
- **TF-IDF** only records **which words and phrases appear** (and how rare they are). Each word is treated separately, and word order is mostly lost.
- **DistilBERT** reads the whole sequence and represents each word **in context**.

Example: the phrases *"no fee required"* and *"pay a fee before starting"* both contain the word **fee**. A TF-IDF model sees the same word in both. DistilBERT can use the surrounding words (*no ... required* versus *pay ... before starting*) to understand that the two sentences mean very different things.

**We are NOT training DistilBERT from scratch.** That would need enormous amounts of text and computing power. Instead we do **fine-tuning**: we start from the pretrained model, add a small classification layer on top (2 outputs: legitimate or fraudulent), and train the whole network for a short time on our labeled job postings so it adapts to this task. Because it already understands language, a small dataset like ours can be enough.

## 5. Tokenization

A transformer cannot read raw text either; it needs numbers. A **tokenizer** converts text into:

- **Tokens:** small pieces of text. Common words are single tokens (`fee`); rare words are split into pieces (`un` + `##believ` + `##able`).
- **Token IDs (`input_ids`):** the number that stands for each token in DistilBERT's fixed vocabulary (about 30,000 entries).
- **Attention mask:** a list of 1s and 0s telling the model which positions are real tokens (1) and which are just **padding** (0).
- **Truncation:** cutting a text that is too long, keeping only the first `max_length` tokens.
- **Padding:** adding filler tokens so all texts in a batch have the same length. We pad each batch only up to its longest text ("dynamic padding"), which is faster than padding everything to 256.

The tokenizer is **part of the pretrained model**. Its vocabulary is fixed, so it does not learn anything from our data (including the validation data). We use the **raw `combined_text`**, with no TF-IDF.

**Why `max_length=256`?** DistilBERT can read at most 512 tokens, and the cost of training grows quickly with length. 256 is a practical compromise that keeps the start of each posting while keeping training time reasonable. The cell below measures what that choice costs us.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
vocab_size_before_tokenizing = len(tokenizer)
print("Tokenizer class:", type(tokenizer).__name__, "| vocabulary size:", vocab_size_before_tokenizing)

In [ ]:
sample_text = X_model_train.iloc[0]
sample_encoding = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH, return_token_type_ids=False)

print("Original text (first 200 characters):")
print(sample_text[:200].replace("\n", " | "), "...")
print()
print("First 20 tokens:   ", tokenizer.convert_ids_to_tokens(sample_encoding["input_ids"][:20]))
print("First 20 token IDs:", sample_encoding["input_ids"][:20])
print("Tokenized length (after truncation):", len(sample_encoding["input_ids"]), "tokens")

In [ ]:
# Padding and the attention mask, shown on two tiny texts
tiny = tokenizer(["no fee required", "pay a fee before starting work today"], padding=True, return_token_type_ids=False)
for text_tokens, mask in zip(tiny["input_ids"], tiny["attention_mask"]):
    print(tokenizer.convert_ids_to_tokens(text_tokens), "->", mask)

In [ ]:
# How long are our postings, in tokens, WITHOUT truncation?
untruncated_lengths = np.array([len(ids) for ids in tokenizer(X_model_train.tolist(), truncation=False,
                                                              return_token_type_ids=False)["input_ids"]])

print("Median length:", int(np.median(untruncated_lengths)), "tokens")
print("90th percentile:", int(np.percentile(untruncated_lengths, 90)), "tokens")
print(f"Postings longer than {MAX_LENGTH} tokens (will be truncated): {(untruncated_lengths > MAX_LENGTH).mean() * 100:.1f}%")
print(f"Postings longer than 512 tokens (DistilBERT's hard limit):   {(untruncated_lengths > 512).mean() * 100:.1f}%")

**Reading these numbers.** With `max_length=256`, every posting longer than 256 tokens is cut off after its first 256 tokens. Because `combined_text` is ordered *title, company profile, description, requirements, benefits*, a truncated posting keeps its beginning and loses its end (typically parts of the requirements and benefits). The percentages printed above show how many postings are affected. This is a **known limitation** of the chosen setting, which we keep unchanged so the experiment stays comparable; it is worth mentioning in the report.

In [ ]:
# Tokenize the model-training and validation text (truncation only; padding is done per batch later)
train_encodings = tokenizer(X_model_train.tolist(), truncation=True, max_length=MAX_LENGTH, return_token_type_ids=False)
validation_encodings = tokenizer(X_validation.tolist(), truncation=True, max_length=MAX_LENGTH, return_token_type_ids=False)

print("Tokenized training examples:  ", len(train_encodings["input_ids"]))
print("Tokenized validation examples:", len(validation_encodings["input_ids"]))
print("Longest tokenized example:", max(len(ids) for ids in train_encodings["input_ids"]), "tokens (limit", MAX_LENGTH, ")")

## 6. PyTorch datasets

The `Trainer` needs a dataset that returns one example at a time with `input_ids`, `attention_mask` and `labels`. This small class does exactly that. A `DataCollatorWithPadding` then pads each batch to its longest example.

In [ ]:
class JobPostingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}   # input_ids, attention_mask
        item["labels"] = int(self.labels[index])
        return item


train_dataset = JobPostingDataset(train_encodings, y_model_train)
validation_dataset = JobPostingDataset(validation_encodings, y_validation)
data_collator = DataCollatorWithPadding(tokenizer)

print("Training dataset size:  ", len(train_dataset))
print("Validation dataset size:", len(validation_dataset))
print("One item has keys:", list(train_dataset[0].keys()))

## 7. The pretrained DistilBERT model

`AutoModelForSequenceClassification` loads the pretrained DistilBERT body and adds a new, **randomly initialized** classification layer with `num_labels=2`. We do **not** freeze the transformer: every layer is fine-tuned.

When loading you will see a report saying the classifier layers are **MISSING** from the checkpoint. That is expected. They are the new layers we just added, and fine-tuning is what trains them. (The "UNEXPECTED" entries belong to BERT's original pretraining head, which we do not need.)

In [ ]:
def load_fresh_model():
    set_seed(SEED)   # same random start for the new classification layer every time
    return AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2)


model_preview = load_fresh_model()
total_parameters = sum(p.numel() for p in model_preview.parameters())
trainable_parameters = sum(p.numel() for p in model_preview.parameters() if p.requires_grad)
print(f"Parameters: {total_parameters:,} total, {trainable_parameters:,} trainable (nothing is frozen)")
print("Output layer:", model_preview.classifier)
del model_preview

## 8. Training settings

Conservative, standard settings for a class project. We do not search over hyperparameters.

| Setting | Value | Meaning |
|---|---|---|
| Learning rate | `2e-5` | very small steps, so we gently adjust the pretrained weights instead of wrecking them |
| Epochs | `2` | the model sees the whole training subset twice |
| Train batch size | `16` | postings processed together |
| Validation batch size | `32` | larger is fine because no training happens |
| Weight decay | `0.01` | mild regularization against overfitting |
| Warmup | first 10% of steps | learning rate rises gradually at the start for stability |
| Seed | `42` | repeatable starting point |

All examples in the model-training subset are used (no silent reduction). Training uses full 32-bit precision (no mixed precision), exactly as in the local notebook, so the methodology is unchanged.

In [ ]:
# The installed Transformers version uses `eval_strategy` (older versions used `evaluation_strategy`).
import inspect
supported = inspect.signature(TrainingArguments.__init__).parameters
for name in ["eval_strategy", "save_strategy", "logging_steps", "warmup_steps", "report_to"]:
    print(f"TrainingArguments supports '{name}':", name in supported)
assert "eval_strategy" in supported and "evaluation_strategy" not in supported

In [ ]:
LEARNING_RATE = 2e-5
EPOCHS = 2
TRAIN_BATCH_SIZE = 16
VALIDATION_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01


def compute_validation_metrics(eval_prediction):
    # Called by the Trainer after every epoch: turn logits into predicted classes and score them
    logits, labels = eval_prediction
    predictions = logits.argmax(axis=-1)
    metrics = calculate_metrics(labels, predictions)
    return {key: metrics[key] for key in ["Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1"]}


def make_trainer(model, train_data, eval_data=None, loss_function=None):
    # Warmup = first 10% of all training steps, rounded up (same rule the Trainer uses for warmup_steps=0.1),
    # written as a whole number so it means the same thing in every library version.
    total_steps = math.ceil(len(train_data) / TRAIN_BATCH_SIZE) * EPOCHS
    warmup_steps = math.ceil(0.1 * total_steps)
    arguments = TrainingArguments(
        output_dir=tempfile.mkdtemp(),           # nothing is saved during training; we save the final model ourselves
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=VALIDATION_BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,               # 10% of all steps
        eval_strategy="epoch" if eval_data is not None else "no",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
        logging_first_step=True,
        seed=SEED,
        report_to="none",
        disable_tqdm=True,
    )
    return Trainer(
        model=model,
        args=arguments,
        train_dataset=train_data,
        eval_dataset=eval_data,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_validation_metrics if eval_data is not None else None,
        compute_loss_func=loss_function,
    )


def evaluate_on_validation(trainer):
    # Predict the validation set, return metrics and the fraud probability of every posting
    output = trainer.predict(validation_dataset)
    logits = output.predictions
    predictions = logits.argmax(axis=-1)
    probabilities = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return calculate_metrics(y_validation, predictions), predictions, probabilities


def epoch_history(trainer):
    # Validation metrics recorded at the end of each epoch (as logged by the Trainer)
    rows = [entry for entry in trainer.state.log_history if "eval_loss" in entry]
    return pd.DataFrame([{"epoch": entry["epoch"], "validation_loss": entry["eval_loss"],
                          "Fraud Precision": entry["eval_Fraud Precision"],
                          "Fraud Recall": entry["eval_Fraud Recall"],
                          "Fraud F1": entry["eval_Fraud F1"]} for entry in rows])


print("Settings:", dict(learning_rate=LEARNING_RATE, epochs=EPOCHS, train_batch_size=TRAIN_BATCH_SIZE,
                        validation_batch_size=VALIDATION_BATCH_SIZE, weight_decay=WEIGHT_DECAY, device=gpu_name))

## 9. Evaluation metrics

For the fraudulent class we report **precision**, **recall** and **F1**, plus overall **accuracy** (which alone would be misleading, because about 95% of postings are legitimate). The Trainer calculates them on the validation subset after every epoch. Fraud F1 is our main summary number, but we always read it together with precision and recall.

## 10. Baseline DistilBERT (no class weighting)

First, an ordinary fine-tuning run. The Trainer prints its training log; we record the time it takes. Only the **model-training subset** is used for learning. The validation subset is used only to measure results after each epoch.

In [ ]:
trainer_baseline = make_trainer(load_fresh_model(), train_dataset, validation_dataset)

start_time = time.time()
trainer_baseline.train()
baseline_training_seconds = time.time() - start_time
print(f"\nBaseline training time: {baseline_training_seconds / 60:.1f} minutes on {device_name}")

In [ ]:
baseline_metrics, baseline_pred, baseline_prob = evaluate_on_validation(trainer_baseline)

print("DistilBERT baseline (VALIDATION results, after the final epoch)")
print(f"  Accuracy:        {baseline_metrics['Accuracy']:.4f}")
print(f"  Fraud precision: {baseline_metrics['Fraud Precision']:.4f}")
print(f"  Fraud recall:    {baseline_metrics['Fraud Recall']:.4f}")
print(f"  Fraud F1-score:  {baseline_metrics['Fraud F1']:.4f}")
print()
print(f"  True Negatives: {baseline_metrics['True Negatives']}   False Positives: {baseline_metrics['False Positives']}")
print(f"  False Negatives: {baseline_metrics['False Negatives']}   True Positives: {baseline_metrics['True Positives']}")
print()
print("Validation metrics after each epoch:")
epoch_history(trainer_baseline).round(4)

## 11. Class-weighted DistilBERT

Only about 4-5% of the postings are fraudulent, so an ordinary model gets little "push" to learn them. A **class-weighted loss** multiplies the training loss for each class: mistakes on the **rare fraud class count more**, mistakes on the common legitimate class count less. We use the standard "balanced" rule, calculated from the **model-training labels** (not typed in):

> weight of a class = (total examples) / (2 x examples in that class)

We train **one** weighted version only. `Trainer` accepts a custom loss through its `compute_loss_func` option, so no subclass is needed. As in earlier models, weighting is **not guaranteed to help**: it usually raises recall and lowers precision.

In [ ]:
count_legitimate = int((y_model_train == 0).sum())
count_fraud = int((y_model_train == 1).sum())
total_examples = count_legitimate + count_fraud

class_weights = torch.tensor([total_examples / (2 * count_legitimate),
                              total_examples / (2 * count_fraud)], dtype=torch.float32)
print(f"Class weights: legitimate = {class_weights[0]:.3f}, fraudulent = {class_weights[1]:.3f} "
      f"(fraud mistakes count about {class_weights[1] / class_weights[0]:.1f}x as much)")


def weighted_loss(outputs, labels, num_items_in_batch=None):
    # Standard cross-entropy, but each class has its own weight
    return torch.nn.functional.cross_entropy(outputs.logits, labels,
                                             weight=class_weights.to(outputs.logits.device))

In [ ]:
trainer_weighted = make_trainer(load_fresh_model(), train_dataset, validation_dataset, loss_function=weighted_loss)

start_time = time.time()
trainer_weighted.train()
weighted_training_seconds = time.time() - start_time
print(f"\nClass-weighted training time: {weighted_training_seconds / 60:.1f} minutes on {device_name}")

In [ ]:
weighted_metrics, weighted_pred, weighted_prob = evaluate_on_validation(trainer_weighted)

print("DistilBERT (Class Weighted) (VALIDATION results, after the final epoch)")
print(f"  Accuracy:        {weighted_metrics['Accuracy']:.4f}")
print(f"  Fraud precision: {weighted_metrics['Fraud Precision']:.4f}")
print(f"  Fraud recall:    {weighted_metrics['Fraud Recall']:.4f}")
print(f"  Fraud F1-score:  {weighted_metrics['Fraud F1']:.4f}")
print()
print(f"  True Negatives: {weighted_metrics['True Negatives']}   False Positives: {weighted_metrics['False Positives']}")
print(f"  False Negatives: {weighted_metrics['False Negatives']}   True Positives: {weighted_metrics['True Positives']}")
print()
print("Validation metrics after each epoch (note: the weighted run's loss uses class weights, so its loss values are not directly comparable with the baseline's):")
epoch_history(trainer_weighted).round(4)

Compare the two tables above (and the per-epoch rows) before drawing conclusions: class weighting usually trades **precision for recall**, but whether it helps overall is decided by the validation numbers, not assumed.

## 12. Select the DistilBERT configuration (validation only)

In [ ]:
distilbert_results = pd.DataFrame([
    {"Model": "DistilBERT", "Configuration": "baseline, no class weights", **baseline_metrics},
    {"Model": "DistilBERT (Class Weighted)", "Configuration": "weighted cross-entropy", **weighted_metrics},
])

comparison_columns = ["Model", "Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1"]
distilbert_results[comparison_columns].round(4)

In [ ]:
best_distilbert = distilbert_results.loc[distilbert_results["Fraud F1"].idxmax()]
selected_name = best_distilbert["Model"]
selected_is_weighted = (selected_name == "DistilBERT (Class Weighted)")

print("Selected DistilBERT configuration:", selected_name)
print(f"  validation fraud F1 {best_distilbert['Fraud F1']:.4f} | precision {best_distilbert['Fraud Precision']:.4f} "
      f"| recall {best_distilbert['Fraud Recall']:.4f}")
print(f"  Missed scams (FN): {int(best_distilbert['False Negatives'])} | False alarms (FP): {int(best_distilbert['False Positives'])}")

**Selection rule:** the configuration with the higher **validation fraud F1** is selected, by code (`idxmax`), and we read its precision/recall balance and the missed-scam / false-alarm counts above alongside it, never accuracy alone. With only about a hundred fraudulent validation postings, a small F1 difference between the two configurations is within the noise, so treat the choice as a sensible default, not a proof.

## 13. Confusion matrix: selected DistilBERT (validation)

In [ ]:
selected_trainer = trainer_weighted if selected_is_weighted else trainer_baseline
selected_pred = weighted_pred if selected_is_weighted else baseline_pred
selected_metrics = weighted_metrics if selected_is_weighted else baseline_metrics

plot_confusion_matrix(y_validation, selected_pred, f"{selected_name} - Validation Set",
                      FIGURES_DIR / "distilbert_validation_confusion_matrix.png")

print("True Negatives :", selected_metrics["True Negatives"])
print("False Positives:", selected_metrics["False Positives"])
print("False Negatives:", selected_metrics["False Negatives"])
print("True Positives :", selected_metrics["True Positives"])

## 14. Training history

The Trainer logged the **training loss** every 50 steps and the **validation loss** at the end of each epoch. Loss measures how wrong the model's probabilities are (lower is better). A validation loss that stops improving while training loss keeps falling would be a sign of overfitting. Only logged values are plotted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)
for ax, (name, trainer) in zip(axes, [("DistilBERT (baseline)", trainer_baseline),
                                      ("DistilBERT (Class Weighted)", trainer_weighted)]):
    history = trainer.state.log_history
    train_points = [(entry["step"], entry["loss"]) for entry in history if "loss" in entry]
    validation_points = [(entry["step"], entry["eval_loss"]) for entry in history if "eval_loss" in entry]

    ax.plot(*zip(*train_points), label="Training loss", color="steelblue")
    ax.plot(*zip(*validation_points), "o-", label="Validation loss (end of epoch)", color="indianred")
    ax.set_title(name + (" - selected" if name.endswith("Weighted)") == selected_is_weighted else ""))
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.legend()

fig.suptitle("DistilBERT Training History")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "distilbert_training_history.png", dpi=150, bbox_inches="tight")
plt.show()

# Also keep the per-epoch validation numbers of both runs as a table (useful for the slides)
epoch_table = pd.concat([epoch_history(trainer_baseline).assign(Configuration="DistilBERT (baseline)"),
                         epoch_history(trainer_weighted).assign(Configuration="DistilBERT (Class Weighted)")],
                        ignore_index=True)
epoch_table.to_csv(METRICS_DIR / "distilbert_epoch_history.csv", index=False)
epoch_table.round(4)

## 15. Four-model validation comparison (optional here)

The first three models' results are **read from `model_validation_comparison.csv`**, the metrics file saved in Milestone 6 on this same validation split. DistilBERT's selected result is added from this notebook. Nothing is typed in by hand.

If you did **not** upload that file in Section 2, this comparison is simply skipped (DistilBERT's own results are still saved); the comparison can then be built locally from the returned metrics.

In [ ]:
distilbert_row = {"Model": "DistilBERT", "Configuration": f"{selected_name}", **selected_metrics}

if HAVE_EARLIER_RESULTS:
    saved_three_models = pd.read_csv(EARLIER_RESULTS_FILE)
    assert len(saved_three_models) == 3, "model_validation_comparison.csv should contain the three earlier models"
    four_models = pd.concat([saved_three_models, pd.DataFrame([distilbert_row])], ignore_index=True)
    display_table = four_models[["Model", "Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1",
                                 "False Positives", "False Negatives"]].round(4)
else:
    four_models = None
    display_table = "model_validation_comparison.csv was not uploaded, so the four-model comparison is skipped."
display_table

In [ ]:
if four_models is not None:
    chart_metrics = ["Fraud Precision", "Fraud Recall", "Fraud F1"]
    bar_colors = ["steelblue", "darkorange", "seagreen", "mediumpurple"]
    positions = np.arange(len(chart_metrics))
    bar_width = 0.2

    fig, ax = plt.subplots(figsize=(9.5, 5))
    for index, (_, row) in enumerate(four_models.iterrows()):
        bars = ax.bar(positions + (index - 1.5) * bar_width, row[chart_metrics].astype(float),
                      bar_width, label=row["Model"], color=bar_colors[index])
        ax.bar_label(bars, fmt="%.3f", fontsize=7.5)

    ax.set_xticks(positions, labels=chart_metrics)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score (fraudulent class)")
    ax.set_title("Four-Model Validation Comparison")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4)
    fig.savefig(FIGURES_DIR / "four_model_validation_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    for _, row in four_models.iterrows():
        print(f"{row['Model']:32s} precision {row['Fraud Precision']:.3f} | recall {row['Fraud Recall']:.3f} | "
              f"F1 {row['Fraud F1']:.3f} | missed scams {int(row['False Negatives'])} | false alarms {int(row['False Positives'])}")
else:
    print("Skipped (no earlier-results file uploaded).")

When the comparison is available, read it as a set of **trade-offs** (precision versus recall, missed scams versus false alarms) rather than looking for a single winner; the final judgement is reserved for the locked test-set evaluation in Milestone 8.

In [ ]:
distilbert_results.to_csv(METRICS_DIR / "distilbert_validation_results.csv", index=False)
saved = ["distilbert_validation_results.csv"]
if four_models is not None:
    four_models.to_csv(METRICS_DIR / "four_model_validation_comparison.csv", index=False)
    saved.append("four_model_validation_comparison.csv")
print("Saved:", saved)

## 16. Save the validation-selected DistilBERT

We save the model that was **selected using validation data** (trained on the model-training subset) with `save_pretrained`, which writes the weights, the model configuration and (for the tokenizer) its vocabulary files.

**Folder naming:** `models/distilbert_validation/` holds this validation-trained model, kept as a record of exactly what the validation numbers describe. The model that will be used for the final test and the web app is the **full-train** version saved to `models/distilbert/` in Section 17, matching how the other models' saved pipelines were built.

To confirm the saved files work, we **reload them from disk** and predict from raw text.

In [ ]:
def predict_fraud_probability(model, tokenizer_to_use, texts, batch_size=16):
    # Raw combined_text in -> probability of fraud out
    model.eval()
    device = next(model.parameters()).device
    probabilities = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer_to_use(texts[start:start + batch_size], truncation=True, max_length=MAX_LENGTH,
                                 padding=True, return_tensors="pt", return_token_type_ids=False).to(device)
        with torch.no_grad():
            logits = model(**batch).logits
        probabilities.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
    return np.array(probabilities)


VALIDATION_MODEL_DIR = MODELS_DIR / "distilbert_validation"
selected_trainer.model.save_pretrained(VALIDATION_MODEL_DIR)
tokenizer.save_pretrained(VALIDATION_MODEL_DIR)
print("Saved files:", sorted(path.name for path in VALIDATION_MODEL_DIR.iterdir()))

reloaded_validation_tokenizer = AutoTokenizer.from_pretrained(VALIDATION_MODEL_DIR)
reloaded_validation_model = AutoModelForSequenceClassification.from_pretrained(VALIDATION_MODEL_DIR).to(device_name)

check_texts = X_validation.iloc[:16].tolist()
reloaded_probabilities = predict_fraud_probability(reloaded_validation_model, reloaded_validation_tokenizer, check_texts)
original_probabilities = predict_fraud_probability(selected_trainer.model, tokenizer, check_texts)
print("Reloaded model reproduces the in-memory model's probabilities:",
      bool(np.allclose(reloaded_probabilities, original_probabilities, atol=1e-4)))
assert np.allclose(reloaded_probabilities, original_probabilities, atol=1e-4)
print("Example fraud probabilities (first 5 validation postings):", reloaded_probabilities[:5].round(3).tolist())

## 17. Retrain the selected configuration on the full `train.csv`

Selection is finished. For the model we will use later, we retrain the **selected configuration** from the pretrained checkpoint on **all of `train.csv`** (model-training + validation subsets). `test.csv` is not included. Same settings, same seed. If the selected configuration is class-weighted, the weights are recomputed from the full training labels with the same rule.

Because this model has now seen the validation postings, we do **not** evaluate it on validation (the scores would be meaningless) and we do not evaluate it on the test set either; that happens in the final milestone. This run takes a bit longer, because there are more examples.

In [ ]:
full_encodings = tokenizer(X.tolist(), truncation=True, max_length=MAX_LENGTH, return_token_type_ids=False)
full_dataset = JobPostingDataset(full_encodings, y)
print("Full-train examples:", len(full_dataset))

if selected_is_weighted:
    count_legitimate_full = int((y == 0).sum())
    count_fraud_full = int((y == 1).sum())
    class_weights = torch.tensor([len(y) / (2 * count_legitimate_full), len(y) / (2 * count_fraud_full)],
                                 dtype=torch.float32)   # the loss function above now uses the full-train weights
    print(f"Recomputed class weights on full train.csv: legitimate = {class_weights[0]:.3f}, fraudulent = {class_weights[1]:.3f}")
    final_loss = weighted_loss
else:
    final_loss = None

# free the memory used by the two validation-stage models first
del trainer_baseline, trainer_weighted, selected_trainer, reloaded_validation_model
import gc
gc.collect()
torch.cuda.empty_cache()

final_trainer = make_trainer(load_fresh_model(), full_dataset, eval_data=None, loss_function=final_loss)
start_time = time.time()
final_trainer.train()
full_training_seconds = time.time() - start_time
print(f"\nFull-train training time: {full_training_seconds / 60:.1f} minutes on {device_name}")

In [ ]:
FINAL_MODEL_DIR = MODELS_DIR / "distilbert"
final_trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved files:", sorted(path.name for path in FINAL_MODEL_DIR.iterdir()))

final_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
final_model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR).to(device_name)

reloaded_final = predict_fraud_probability(final_model, final_tokenizer, check_texts)
in_memory_final = predict_fraud_probability(final_trainer.model, tokenizer, check_texts)
assert np.allclose(reloaded_final, in_memory_final, atol=1e-4)
print("PASS: the reloaded full-train model reproduces the in-memory model's predictions")
print("Example fraud probabilities (first 5 validation postings - these were part of full training, so this is NOT an evaluation):",
      reloaded_final[:5].round(3).tolist())

## 18. Sanity-check with demonstration examples

> **Hand-written demonstration examples only. They are not test data and are not used for any decision.**

The same three examples as in the other notebooks. We show the final full-train model and, for comparison, the validation-trained model.

In [ ]:
demo_examples = pd.DataFrame([
    {
        "name": "Professional posting (demo)",
        "title": "Software Engineer",
        "company_profile": "Northwind Analytics is a data company serving hospitals and universities. "
                           "We build tools that help researchers understand their data.",
        "description": "We are looking for a software engineer to join our platform team. You will design, "
                       "build and test backend services in Python, review code with teammates, and "
                       "participate in on-call rotation.",
        "requirements": "Bachelor's degree in computer science or equivalent experience. 3 years of Python "
                        "and SQL. Experience with automated testing and version control.",
        "benefits": "Health, dental and vision insurance, 401(k) match, paid vacation.",
    },
    {
        "name": "Suspicious posting (demo)",
        "title": "Work From Home - Earn $5000 Weekly!!!",
        "company_profile": "",
        "description": "No experience needed! Earn thousands every week from home. Limited spots. "
                       "Send your full name, address, date of birth and bank account details to "
                       "start immediately. A small registration fee is required.",
        "requirements": "Must have a bank account and be able to pay the registration fee.",
        "benefits": "",
    },
    {
        "name": "Very short posting (demo)",
        "title": "Data Entry Clerk",
        "company_profile": "",
        "description": "Data entry. Work from home. Apply now.",
        "requirements": "",
        "benefits": "",
    },
])

demo_text = create_combined_text(demo_examples).tolist()
final_demo_prob = predict_fraud_probability(final_model, final_tokenizer, demo_text)
reloaded_validation_model = AutoModelForSequenceClassification.from_pretrained(VALIDATION_MODEL_DIR).to(device_name)
validation_demo_prob = predict_fraud_probability(reloaded_validation_model, reloaded_validation_tokenizer, demo_text)

print(f"{'Example':32s} {'Final (full-train) model':36s} Validation-trained model")
for name, final_p, validation_p in zip(demo_examples["name"], final_demo_prob, validation_demo_prob):
    final_label = "Potentially Fraudulent" if final_p >= 0.5 else "Likely Legitimate"
    validation_label = "Potentially Fraudulent" if validation_p >= 0.5 else "Likely Legitimate"
    print(f"{name:32s} {final_label:22s} ({final_p:.3f})     {validation_label:22s} ({validation_p:.3f})")

## 19. Validation checks

Each `assert` stops the notebook with an error if a check fails.

In [ ]:
# 1. train.csv is unchanged, and no test file was ever present in this session
assert file_fingerprint(TRAIN_FILE) == train_csv_fingerprint_at_start == EXPECTED["train_csv_md5"]
assert not any(path.name.lower().startswith("test") for path in Path(".").glob("*.csv"))
print("PASS: train.csv is unchanged and identical to the local one; no test file exists in this session")

# 2. No test text or labels were loaded
assert not any(name in globals() for name in ("X_test", "y_test", "test_df", "test", "test_rows"))
print("PASS: no test text, labels or row counts were loaded")

# 3. Internal split matches the local project exactly, with no text overlap
assert split_matches_previous
assert len(set(X_model_train) & set(X_validation)) == 0
print("PASS: same model-training rows and validation rows as the local project; 0 overlapping combined_text values")

# 4. The tokenizer is the fixed pretrained one: no vocabulary was added from our data
assert len(tokenizer) == vocab_size_before_tokenizing
assert len(tokenizer) == AutoTokenizer.from_pretrained(CHECKPOINT).__len__()
print("PASS: tokenizer vocabulary is unchanged and identical to the pretrained checkpoint")

# 5. During selection, training used only the model-training subset and evaluation only the validation subset
assert len(train_dataset) == len(X_model_train) == EXPECTED["model_train_rows"]
assert len(validation_dataset) == len(X_validation) == EXPECTED["validation_rows"]
print("PASS: selection runs used only the internal model-training subset (and were scored on validation)")

# 6. All metrics come from validation predictions
for metrics in (baseline_metrics, weighted_metrics):
    assert (metrics["True Negatives"] + metrics["False Positives"]
            + metrics["False Negatives"] + metrics["True Positives"]) == len(y_validation)
print("PASS: confusion-matrix counts of both configurations add up to the validation size")

# 7. The final model used all of train.csv and nothing else
assert x_is_all_of_train_csv and len(full_dataset) == len(X) == EXPECTED["train_csv_rows"]
print("PASS: the final model was retrained on the full train.csv only (no test data)")

# 8. Saved models and tokenizers reload and predict
for folder in (VALIDATION_MODEL_DIR, FINAL_MODEL_DIR):
    names = {path.name for path in folder.iterdir()}
    assert "config.json" in names and any(n.endswith(".safetensors") for n in names)
    assert any("tokenizer" in n or n == "vocab.txt" for n in names)
assert final_demo_prob.shape == (3,) and validation_demo_prob.shape == (3,)
print("PASS: both saved model folders contain weights, config and tokenizer files and reload successfully")

# 9. Metrics files match the notebook results
metric_names = ["Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1"]
saved_distilbert = pd.read_csv(METRICS_DIR / "distilbert_validation_results.csv")
assert np.allclose(saved_distilbert[metric_names].values, distilbert_results[metric_names].astype(float).values)
if four_models is not None:
    saved_four = pd.read_csv(METRICS_DIR / "four_model_validation_comparison.csv")
    assert np.allclose(saved_four[metric_names].values, four_models[metric_names].astype(float).values)
    assert len(saved_four) == 4
print("PASS: the metrics CSV files match the notebook results")

# 10. Figures exist and are non-empty
figure_names = ["distilbert_validation_confusion_matrix.png", "distilbert_training_history.png"]
if four_models is not None:
    figure_names.append("four_model_validation_comparison.png")
for figure_name in figure_names:
    assert (FIGURES_DIR / figure_name).stat().st_size > 0
print("PASS: all expected figure files were saved:", figure_names)

## 20. Summary (numbers printed by code)

In [ ]:
print("DEVICE:", device_name, "-", gpu_name)
print(f"INTERNAL SPLIT: {len(X_model_train):,} model-training rows (fraud {int(y_model_train.sum())}), "
      f"{len(X_validation):,} validation rows (fraud {int(y_validation.sum())})")
print(f"TOKENIZATION: {CHECKPOINT}, max_length={MAX_LENGTH}, {len(train_dataset):,} training and {len(validation_dataset):,} validation examples")
print(f"TRAINING: {EPOCHS} epochs, learning rate {LEARNING_RATE}, train batch {TRAIN_BATCH_SIZE}, validation batch {VALIDATION_BATCH_SIZE}")
print(f"TIME: baseline {baseline_training_seconds / 60:.1f} min | weighted {weighted_training_seconds / 60:.1f} min | "
      f"full-train retraining {full_training_seconds / 60:.1f} min")
print()
print(distilbert_results[comparison_columns].round(4).to_string(index=False))
print("Selected:", selected_name)
print()
if four_models is not None:
    print(four_models[["Model", "Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1"]].round(4).to_string(index=False))
    print()
print("TEST SET: test.csv was never uploaded, read or evaluated in this notebook.")

## 21. Package the results for download

Colab's storage is temporary, so we bundle what you need into **two zip files** and start their download:

| File | Contents | Where it goes in your local project |
|---|---|---|
| `jobguard_m7_results.zip` | all figures, metrics CSVs and a run-info file (small) | unzip in the **project root**: it creates `results/figures/...` and `results/metrics/...` |
| `jobguard_m7_model_distilbert.zip` | the **final selected DistilBERT** (retrained on all of `train.csv`) and its tokenizer | unzip in the **project root**: it creates `models/distilbert/` |

The validation-trained model (`models/distilbert_validation/`) is only a record of what the validation numbers describe; it is **not** downloaded by default (to save time). Set `INCLUDE_VALIDATION_MODEL = True` below if you want it as well.

The run-info file records the GPU, library versions, settings and the **measured** training times, so compute notes in the report use real numbers.

In [ ]:
# A small record of exactly how this run was made (all values come from the code above)
run_info = {
    "gpu_name": gpu_name,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "checkpoint": CHECKPOINT,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "validation_batch_size": VALIDATION_BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "selected_configuration": selected_name,
    "final_model_trained_on_rows": len(full_dataset),
    "baseline_training_minutes": round(baseline_training_seconds / 60, 2),
    "weighted_training_minutes": round(weighted_training_seconds / 60, 2),
    "full_train_training_minutes": round(full_training_seconds / 60, 2),
    "train_csv_md5": train_csv_fingerprint_at_start,
    "model_train_index_md5": actual["model_train_index_md5"],
    "validation_index_md5": actual["validation_index_md5"],
    "test_csv_used": "no",
}
pd.DataFrame({"item": list(run_info), "value": list(run_info.values())}).to_csv(
    METRICS_DIR / "distilbert_colab_run_info.csv", index=False)

INCLUDE_VALIDATION_MODEL = False   # set True to also download models/distilbert_validation/

results_zip = shutil.make_archive("jobguard_m7_results", "zip", root_dir=OUTPUT_ROOT, base_dir="results")
model_zip = shutil.make_archive("jobguard_m7_model_distilbert", "zip", root_dir=OUTPUT_ROOT, base_dir="models/distilbert")
zips_to_download = [results_zip, model_zip]
if INCLUDE_VALIDATION_MODEL:
    zips_to_download.append(shutil.make_archive("jobguard_m7_model_distilbert_validation", "zip",
                                                root_dir=OUTPUT_ROOT, base_dir="models/distilbert_validation"))

import zipfile
for zip_path in zips_to_download:
    with zipfile.ZipFile(zip_path) as archive:
        names = [n for n in archive.namelist() if not n.endswith("/")]
    print(f"{zip_path}  ({Path(zip_path).stat().st_size / 1024 ** 2:.1f} MB)")
    for name in names:
        print("   ", name)

In [ ]:
# Start the downloads (your browser may ask permission to download multiple files; please allow it)
try:
    from google.colab import files
    for zip_path in zips_to_download:
        files.download(zip_path)
except ImportError:
    print("Not running in Colab: the zip files are in the current folder:", zips_to_download)

## What to do next (back in your local project)

1. Unzip both files **in the project root** (the folder that contains `data/`, `models/`, `results/`, `src/`), so the contents land in `results/figures/`, `results/metrics/` and `models/distilbert/`.
2. Tell the assistant that the Colab results are in place. The interpretation of the results (selection reasoning, four-model trade-offs, what we learned) is written from the **real downloaded numbers**, not before.
3. **Do not start Milestone 8** (the final locked test-set evaluation) until the Milestone 7 results have been reviewed.